In [1]:
import string
import nltk
import random
import abc
import sklearn
import numpy as np
from sklearn.metrics import (precision_score, recall_score, f1_score, confusion_matrix,
                                roc_auc_score, precision_recall_curve, auc)
nltk.download('stopwords')

stop_words = set(nltk.corpus.stopwords.words('english'))

[nltk_data] Downloading package stopwords to /home/fbb/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
def data_extraction(file_path="Data/SMSSpamCollection"):
    """
    Given a filepath, extract the data within the file
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = [line.strip() for line in f]
            return data
    except FileNotFoundError:
        print(f"ERROR: File not found at {filepath}")
        return []

In [3]:
def data_processing(data, stop_words):
    """
    Given the data, return tuple of data and label
    """
    finished_data =[]
    for sms in data:
        label, text = sms.split("\t")
        label = 1 if label == "spam" else 0
        text = text.lower()
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = text.replace('\x92', "'")
        words = text.split()
        words = [word for word in words if word not in stop_words]
        text = " ".join(words)
        #finished_data.append((label, text))
        finished_data.append((text, label))
    return finished_data

In [10]:
class negative_selection():

    def __init__(self, k=3, max_iters=200, seed=42):
        self.k = k
        self.detectors = set()
        self.max_iters = max_iters
        self.rng = np.random.default_rng(seed)

    def extract_kgrams(self,text):
        """
        Returns a list of all possible k length substrings out of the text
        """
        k = self.k
        return [text[i:i+k] for i in range(len(text) - k + 1)]

    def encode(self, train):
        """
        Returns all possible ham_k_grams out of the training corpus
        """
        k_grams = set()
        for sms, label in zip(train[0], train[1]):
            if label == '1':
                continue
            if len(sms) < self.k:
                continue
            k_grams.update(self.extract_kgrams(sms))
        print(f"Extracted {len(k_grams)} ham k-grams for k={self.k}")
        return k_grams

    def encode_spam(self, data):
        """
        Extract k-grams only from spam messages in the dataset.
        """
        spam_k_grams = set()
        for sms, label in zip(data[0], data[1]):
            if label == '1' and len(sms) >= self.k:
                spam_k_grams.update(self.extract_kgrams(sms))
        print(f"Extracted {len(spam_k_grams)} spam k-grams for k={self.k}")
        return spam_k_grams

    def evaluate_spam_detection(self, y_true, y_pred):
       
        # Precision, Recall, F1 for spam class (class=1)
        precision_spam = precision_score(y_true, y_pred, zero_division=0)
        recall_spam = recall_score(y_true, y_pred, zero_division=0)
        f1_spam = f1_score(y_true, y_pred, zero_division=0)

    # Macro F1 (averages over both classes equally)
        macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    # Confusion matrix to get false positive rate on harmless messages (class 0)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        fpr_harmless = fp / (fp + tn) if (fp + tn) > 0 else 0

        return (precision_spam, recall_spam, f1_spam, macro_f1, (tn, fp, fn, tp), fpr_harmless)

    def average_eval_results(self, results):
        n = len(results)
        sum_precision = 0.0
        sum_recall = 0.0
        sum_f1 = 0.0
        sum_macro_f1 = 0.0
        sum_tn = 0
        sum_fp = 0
        sum_fn = 0
        sum_tp = 0
        sum_fpr = 0.0

        for (precision_spam, recall_spam, f1_spam, macro_f1, (tn, fp, fn, tp), fpr_harmless) in results:
            sum_precision += precision_spam
            sum_recall += recall_spam
            sum_f1 += f1_spam
            sum_macro_f1 += macro_f1
            sum_tn += tn
            sum_fp += fp
            sum_fn += fn
            sum_tp += tp
            sum_fpr += fpr_harmless

        avg_precision = sum_precision / n
        avg_recall = sum_recall / n
        avg_f1 = sum_f1 / n
        avg_macro_f1 = sum_macro_f1 / n
        avg_fpr = sum_fpr / n

        return (avg_precision, avg_recall, avg_f1, avg_macro_f1, (sum_tn, sum_fp, sum_fn, sum_tp), avg_fpr)
    
    def detect_with_r_continguous_bits(self, texts):
        """
        Creates all possible substrings, then matches them against the detector set
        """
        detectors = self.detectors
        y_pred = []
        for text in texts:
            substrings = self.extract_kgrams(text)
            if any(substring in detectors for substring in substrings):
                y_pred.append(1)
            else:
                y_pred.append(0)
        return np.array(y_pred)

    def detect_with_exact_match(self, messages):
        """
        Checks if the given message contains any detectors, if yes it is flagged as spam
        """
        detectors = self.detectors
        y_pred = []
        for message in messages:
            if any(d in message for d in detectors):
                y_pred.append(1)
            else:    
                y_pred.append(0)
        return np.array(y_pred)

    def hamming_distance(self, s1, s2):
        """Compute Hamming distance between equal length strings."""
        assert len(s1) == len(s2)
        return sum(ch1 != ch2 for ch1, ch2 in zip(s1, s2))

    def detect_with_r_mismatches(self, texts, r=1):
        """
        Detect spam with detectors allowing up to r mismatches
        """
        detectors = self.detectors
        k = self.k
        y_pred = []
        for text in texts:
            substrings = self.extract_kgrams(text)
            detected = False
            for substring in substrings:
                for detector in detectors:
                    if self.hamming_distance(substring, detector) <= r:
                        detected = True
                        break
                if detected:
                    break
            y_pred.append(1 if detected else 0)
        return np.array(y_pred)

    def average_matches_per_message(self, texts, labels, method='exact'):
        """
        Compute average detector matches per spam and ham message.
        method controls detection routine: 'exact', 'substring', 'hamming'
        Returns: avg_matches_spam, avg_matches_ham
        """
        detectors = self.detectors
        matches_per_spam = []
        matches_per_ham = []

        if method == 'exact':
            for text, label in zip(texts, labels):
                
                count = sum(d in text for d in detectors)
                if label == 1:
                    matches_per_spam.append(count)
                else:
                    matches_per_ham.append(count)
        elif method == 'substring':
            for text, label in zip(texts, labels):
                substrings = self.extract_kgrams(text)
                count = sum(substring in detectors for substring in substrings)
                if label == '1':
                    matches_per_spam.append(count)
                else:
                    matches_per_ham.append(count)
        elif method == 'hamming':
            for text, label in zip(texts, labels):
                substrings = self.extract_kgrams(text)
                count = 0
                for substring in substrings:
                    for detector in detectors:
                        if self.hamming_distance(substring, detector) <= 1:  # or r param if exposed
                            count += 1
                            break
                if label == '1':
                    matches_per_spam.append(count)
                else:
                    matches_per_ham.append(count)
        else:
            raise ValueError("Detection method not recognized")
            
        avg_spam = np.mean(matches_per_spam) if matches_per_spam else np.nan()
        avg_ham = np.mean(matches_per_ham) if matches_per_ham else np.nan()
        return avg_spam, avg_ham

    def random_string(self, alphabet):
        """
        Creates a random string to match against k_grams in data base
        """
        return ''.join(self.rng.choice(alphabet, size=self.k))
        #return ''.join(random.choice(alphabet) for _ in range(self.k))
            
    
    def generate_detectors(self, k_grams, alphabet=None):
        """
        Out of the given alphabet, or default, it creates strings, which are matched against the k_grams. 
        If not within the k_grams the string is added as a detector.
        """
        if alphabet == None:
            alphabet = list(string.ascii_lowercase + string.digits)
        detectors = self.detectors
        cand = self.random_string(alphabet)
        if cand not in k_grams:
            detectors.add(cand)
        return detectors
        
    def generate_detectors_batched(self, k_grams, alphabet=None, batch_size=100):
        """
        Out of the given alphabet, or default, it creates strings, which are matched against the k_grams. 
        If not within the k_grams the string is added as a detector.
        """
        if alphabet == None:
            alphabet = list(string.ascii_lowercase + string.digits)
        detectors = self.detectors
        candidates = set()
        attempts = 0
        max_attempts = batch_size * 10
        while len(candidates) < batch_size and attempts < max_attempts:
            cand = self.random_string(alphabet)
            if (cand not in k_grams) and (cand not in candidates) and (cand not in detectors):
                candidates.add(cand)
            attempts += 1
        
        detectors.add(cand)
        return detectors

    def generate_detectors_guided(self, spam_k_grams, ham_k_grams, batch_size=100):
        """
        Generate detectors by sampling from spam k-grams that are not present in ham k-grams.
        """
        detectors = self.detectors
        candidates = set()
        spam_k_grams_list = list(spam_k_grams)
        attempts = 0
        max_attempts = batch_size * 10
    
        while len(candidates) < batch_size and attempts < max_attempts:
            cand = self.rng.choice(spam_k_grams_list)
            # Add candidate only if not in ham_k_grams, and not already in detectors/candidates
            if (cand not in ham_k_grams) and (cand not in detectors) and (cand not in candidates):
                candidates.add(cand)
            attempts += 1
    
        detectors.update(candidates)
        return detectors


    def run(self, train, test, seed, split_n, detection_method='exact', batch_size=1, guided=False, patience=2, min_delta=1e-4):
        X_train, Y_train = train
        X_train, X_val, Y_train, Y_val = sklearn.model_selection.train_test_split(X_train, Y_train, test_size=0.1, random_state = seed, stratify=Y_train)
        X_test, Y_test = test
        average_val = []
        modulo_val = self.max_iters / 10
        best_f1 = -np.inf
        patience_counter = 0
        
        if detection_method == 'exact':
            detection = self.detect_with_exact_match
        elif detection_method == 'substring':
            detection = self.detect_with_r_continguous_bits
        elif detection_method == 'hamming':
            detection = lambda texts: self.detect_with_r_mismatches(texts, r=1)
        else:
            print("No function known, proceeding with exact matching as backup!")
            detection = self.detect_with_exact_match
            
        
        train = X_train, Y_train
        val = X_val, Y_val
        train = np.array(train)
        val = np.array(val)
        ham_k_grams = self.encode(train)
        spam_k_grams = self.encode_spam(train) if guided else None
        for i in range(self.max_iters):
            if guided:
                self.detectors = self.generate_detectors_guided(spam_k_grams, ham_k_grams, batch_size=batch_size)
            else:
                self.detectors = self.generate_detectors(ham_k_grams)
           
            
            if i % modulo_val == 0:
                Y_pred = detection(X_val)
                values = self.evaluate_spam_detection(Y_val, Y_pred)
                average_val.append(values)
                precision, recall, f1_spam, macro_f1, confusion_matrix, fpr = values
                avg_matches_spam, avg_matches_ham = self.average_matches_per_message(X_val, Y_val, detection_method)
                print(f"{self.k:3d} | {i:10} | VAL  | {precision:9.3f} | {recall:7.3f} | {f1_spam:7.3f} | {macro_f1:8.3f} | {fpr:12.3f} | {len(self.detectors):9.1f} | {avg_matches_spam:14.2f} | {avg_matches_ham:3.2f}")
                # Early stopping check based on f1_spam
                if f1_spam - best_f1 > min_delta:
                    best_f1 = f1_spam
                    patience_counter = 0  # reset counter
                else:
                    patience_counter += 1

                if patience_counter >= patience:
                    print(f"Early stopping triggered at iteration {i} with best Spam F1: {best_f1:.4f}")
                    break

        average_val = self.average_eval_results(average_val)
        Y_pred = detection(X_test)
        test_values = self.evaluate_spam_detection(Y_test, Y_pred)
        
        return average_val, test_values, len(self.detectors)

In [8]:
def test_varying_k(ns_class, train_data, test_data, seed=42, detection_met = 'exact', split_n=5, k_values=None, max_iters=200, batch_size=100, guided=False):
    """
    Test the negative_selection classifier for multiple k values and print aggregated results.

    Parameters:
        ns_class: The negative_selection class (uninitialized)
        train_data: Tuple (X_train, Y_train) - np.array of texts and labels
        test_data: Tuple (X_test, Y_test)
        seed: Random seed for reproducibility
        split_n: Number of folds in KFold cross-validation
        k_values: List of k values to test (e.g. [2,3,4,5])

    Prints evaluation metrics for each k, averaged over all iterations and folds.
    """
    if k_values is None:
        k_values = [2, 3, 4, 5]
    if max_iters is None:
        max_iters = [20, 50, 100, 200]

    print(f"{'k':>3} | {'Iterations':>10} | {'Mode':>4} | {'Precision':>9} | {'Recall':>7} | {'F1 Spam':>7} | {'Macro F1':>8} | {'FPR Harmless':>12} | {'Detectors':>9} | {'Avg match:spam':>14} | {'ham':>3}")
    print('-' * 118)

    for k in k_values:
        for max_iter in max_iters:
            model = ns_class(k=k, max_iters=max_iter)
            avg_val_metrics, test_metrics, avg_detectors = model.run(train_data, test_data, seed, split_n, detection_method=detection_met, guided=guided)
        
            # average_val_metrics is a list over max_iters of tuples, average those
            precision, recall, f1_spam, macro_f1, (tn, fp, fn, tp), fpr = avg_val_metrics
            print(f"{k:3d} | {max_iter:10} | VAL  | {precision:9.3f} | {recall:7.3f} | {f1_spam:7.3f} | {macro_f1:8.3f} | {fpr:12.3f} | {avg_detectors:9.1f}")
            
            precision, recall, f1_spam, macro_f1, (tn, fp, fn, tp), fpr = test_metrics
            print(f"{k:3d} | {max_iter:10} | TEST | {precision:9.3f} | {recall:7.3f} | {f1_spam:7.3f} | {macro_f1:8.3f} | {fpr:12.3f} | {avg_detectors:9.1f}")
            print('-' * 118)
    print()

In [6]:
def experiment(k_values, max_iters, seed = 42, detection_met = 'exact', batch_size=100, guided=False):
    file_path = "Data/SMSSpamCollection"

    data = data_extraction(file_path)

    data = data_processing(data, stop_words)
    X, Y = zip(*data)

    X = list(X)
    Y = list(Y)

    X = np.array(X)
    Y = np.array(Y)

    X_train, X_test, Y_train, Y_test = sklearn.model_selection.train_test_split(X, Y, test_size=0.1, random_state = seed, stratify=Y)

    train = X_train, Y_train
    test = X_test, Y_test

    test_varying_k(negative_selection, train, test, seed=seed, detection_met=detection_met, split_n=5, k_values=k_values, max_iters=max_iters, guided=guided)

In [11]:
k_values = [2, 3, 4]
max_iters = [20, 50, 100, 200, 500, 1000, 1500, 10000, 20000]

max_iters = [10000]
experiment(k_values, max_iters, 42, 'exact', guided = True)

  k | Iterations | Mode | Precision |  Recall | F1 Spam | Macro F1 | FPR Harmless | Detectors | Avg match:spam | ham
----------------------------------------------------------------------------------------------------------------------
Extracted 905 ham k-grams for k=2
Extracted 968 spam k-grams for k=2
  2 |          0 | VAL  |     0.000 |   0.000 |   0.000 |    0.464 |        0.000 |       1.0 |           0.00 | 0.00
  2 |       1000 | VAL  |     0.881 |   0.881 |   0.881 |    0.931 |        0.018 |     258.0 |           3.87 | 0.02
  2 |       2000 | VAL  |     0.881 |   0.881 |   0.881 |    0.931 |        0.018 |     258.0 |           3.87 | 0.02
  2 |       3000 | VAL  |     0.881 |   0.881 |   0.881 |    0.931 |        0.018 |     258.0 |           3.87 | 0.02
Early stopping triggered at iteration 3000 with best Spam F1: 0.8806
  2 |      10000 | VAL  |     0.660 |   0.660 |   0.660 |    0.814 |        0.014 |     258.0
  2 |      10000 | TEST |     0.931 |   0.893 |   0.912 |   